<a href="https://colab.research.google.com/github/gautamkr1876/AIML_ClassNotes/blob/main/8.%20Agentic%20AI%20Systems/13.%20Parsing%20Complex%20Documents/L13_Parsing_Complex_Documents.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Parsing Complex Documents for AI Systems

## What You’ll Build

Modern AI applications often work with documents such as PDFs, invoices, reports, scanned forms, and spreadsheets. Before an LLM can understand any of these documents, the information needs to be extracted, structured, and prepared for AI processing.

This process is called **document ingestion**.

Document ingestion is a critical part of applications such as:

- RAG systems
- AI agents
- Enterprise search
- Document intelligence platforms
- AI assistants with file uploads
- Knowledge management systems

A document pipeline can determine how well an AI system understands the information inside a document. Poor parsing can lead to missing information, broken tables, incorrect relationships between text sections, and unreliable answers.

## The Project: FinBot 2.0

You’ll build **FinBot 2.0**, an enterprise-style invoice intelligence pipeline.

The system will process a folder containing different types of documents, including:

- Native PDFs
- Scanned documents
- Images
- Handwritten notes
- Memos
- Documents containing tables

The pipeline will transform these documents into **LLM-ready knowledge** that can be searched and retrieved by an AI system.

The final system will expose this knowledge as a **tool for an AI agent**, allowing users to ask procurement-related questions in natural language.

For example:

> "Which suppliers have outstanding invoices above ₹10 lakh?"

Or:

> "Show me the invoices from Acme Corp that are overdue."

The AI agent can use the parsed document data to find the relevant information and generate an answer.

## What You’ll Learn

By the end of this lesson, you’ll understand how modern AI systems process complex documents.

### 1. How AI Systems Read PDFs

You’ll explore the different stages involved in turning a document into usable data:

- PDF parsing
- Text extraction
- OCR
- Table extraction
- Layout understanding
- Document structure detection

### 2. The Modern Document Parsing Ecosystem

You’ll explore different approaches to document parsing, including:

- Open-source parsing tools
- AI-native document parsers
- Enterprise cloud document intelligence services
- API-based document processing

### 3. API-First Document Ingestion

You’ll learn how to design document pipelines around APIs so that documents can be processed reliably and integrated into larger AI applications.

### 4. Batch Processing at Scale

Real-world systems may need to process thousands or millions of documents.

You’ll learn how batch processing can be used to build scalable document ingestion pipelines.

### 5. Layout-Aware Chunking

Traditional text chunking can destroy important relationships inside a document.

For example, separating a table from its heading can make the table difficult for an LLM to understand.

You’ll learn how **layout-aware chunking** preserves the structure and context of documents, improving downstream retrieval and RAG performance.

### 6. From Documents to AI Agents

Finally, you’ll connect the entire pipeline:

**Documents → Parsing → Structured Content → Chunking → Embeddings → Retrieval → Agent Tool**

The result is a complete document intelligence pipeline that can transform messy real-world documents into information an AI agent can understand and use.

## The End Goal

By completing this project, you’ll be able to design and build a document processing pipeline for an LLM application—not just extract text from a PDF.

You’ll understand how raw documents become searchable knowledge and how that knowledge can ultimately power intelligent AI applications.

That is PyMuPDF. Fast, no cost, and for clean native PDFs it is genuinely enough. But notice: the table structure is flattened into text. We do not know which cell was a header, which was a total. For a RAG system that needs to answer *'what was the total for the Wireless Mouse line?'*, this flattened output is fragile. Now watch what happens when we use a parser designed for AI pipelines.

## Section 0: Setup

Before we start building the document pipeline, we need to prepare our environment.

The setup has three main parts:

1. **Install the required Python packages**
2. **Install any required system dependencies**
3. **Generate a sample document corpus** that we’ll use throughout the project

After that, we’ll configure the API keys required by the services we’ll use.

Once the environment is ready, we can start processing documents and building our AI pipeline.

In [ ]:
!pip -q install pymupdf pytesseract "unstructured[pdf]" llama-parse reportlab Pillow transformers torch sentence-transformers faiss-cpu groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 12.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.2/542.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 24.4 MB/s eta 0:00:00


In [ ]:
!apt-get -qq install -y tesseract-ocr poppler-utils > /dev/null
print('System dependencies ready.')

System dependencies ready.


In [ ]:
# Generate the invoice corpus that FinBot 2.0 will ingest.
import os
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
from PIL import Image, ImageDraw, ImageFont

os.makedirs('invoices', exist_ok=True)
styles = getSampleStyleSheet()

def _font(size=20, italic=False):
    path = '/usr/share/fonts/truetype/dejavu/DejaVuSans-Oblique.ttf' if italic else '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'
    try:
        return ImageFont.truetype(path, size)
    except Exception:
        return ImageFont.load_default()

def make_native_invoice(path):
    doc = SimpleDocTemplate(path, pagesize=letter)
    data = [['SKU', 'Description', 'Qty', 'Unit Price', 'Total'],
            ['A101', 'USB-C Cable 1m', '50', '3.20', '160.00'],
            ['A102', 'Wireless Mouse', '20', '12.50', '250.00'],
            ['A103', 'Mechanical Keyboard', '10', '48.00', '480.00'],
            ['A104', 'Laptop Stand Aluminum', '15', '22.00', '330.00']]
    t = Table(data, style=[('BACKGROUND',(0,0),(-1,0),colors.grey),
                           ('TEXTCOLOR',(0,0),(-1,0),colors.whitesmoke),
                           ('GRID',(0,0),(-1,-1),0.5,colors.black)])
    story = [Paragraph('Vendor: NovaTech Supplies Pvt Ltd', styles['Title']),
             Paragraph('Invoice No: NT-2024-0912  |  Date: 2024-09-18', styles['Normal']),
             Paragraph('Buyer: Amazon India Fulfillment Center, Hyderabad', styles['Normal']),
             Spacer(1, 18), t, Spacer(1, 12),
             Paragraph('Subtotal: 1220.00 | Tax (18%): 219.60 | Grand Total: 1439.60 USD', styles['Normal']),
             Paragraph('Payment Terms: Net 30. Late fees apply after due date.', styles['BodyText'])]
    doc.build(story)

def make_multipage_invoice(path):
    doc = SimpleDocTemplate(path, pagesize=letter)
    p1 = [['SKU','Description','Qty','Total'],['B201','HDMI Cable','100','250.00'],['B202','Ethernet Cable','80','160.00'],['B203','Power Adapter','40','400.00']]
    p2 = [['SKU','Description','Qty','Total'],['B204','Cable Organizer','30','90.00'],['B205','Surge Protector','25','175.00']]
    t1 = Table(p1, style=[('GRID',(0,0),(-1,-1),0.5,colors.black),('BACKGROUND',(0,0),(-1,0),colors.lightgrey)])
    t2 = Table(p2, style=[('GRID',(0,0),(-1,-1),0.5,colors.black),('BACKGROUND',(0,0),(-1,0),colors.lightgrey)])
    story = [Paragraph('Vendor: CableWorld Inc | Invoice CW-88', styles['Title']),
             Spacer(1,12), t1, PageBreak(),
             Paragraph('Continued from previous page', styles['Italic']),
             Spacer(1,12), t2]
    doc.build(story)

def make_scanned_invoice(path):
    img = Image.new('RGB', (900, 500), 'white'); d = ImageDraw.Draw(img)
    d.text((30, 30), 'BharatSupply Traders', fill='black', font=_font(28))
    d.text((30, 80), 'Invoice No: BS-2024-441   Date: 2024-08-11', fill='black', font=_font(18))
    d.text((30, 120), 'Buyer: Flipkart Warehouse, Bengaluru', fill='black', font=_font(18))
    d.text((30, 180), 'Item                Qty     Total', fill='black', font=_font(18))
    d.text((30, 210), 'Packing Tape         200     600.00', fill='black', font=_font(18))
    d.text((30, 240), 'Bubble Wrap Roll     50      450.00', fill='black', font=_font(18))
    d.text((30, 270), 'Corrugated Boxes     300    1200.00', fill='black', font=_font(18))
    d.text((30, 340), 'Grand Total: 2250.00 INR', fill='black', font=_font(20))
    d.text((30, 400), 'Terms: Payment within 15 days.', fill='black', font=_font(16))
    img.save(path)

def make_handwritten_note(path):
    img = Image.new('RGB', (700, 180), 'white'); d = ImageDraw.Draw(img)
    d.text((25, 60), 'Approved by Rohan on 12 Sept', fill='black', font=_font(30, italic=True))
    img.save(path)

def make_mixed_memo(path):
    doc = SimpleDocTemplate(path, pagesize=letter)
    data = [['Vendor','Invoices','Spend (USD)'],['NovaTech','12','17,200'],['CableWorld','8','9,400'],['BharatSupply','5','6,100']]
    t = Table(data, style=[('GRID',(0,0),(-1,-1),0.5,colors.black),('BACKGROUND',(0,0),(-1,0),colors.lightgrey)])
    story = [Paragraph('Amazon Procurement Memo, Q3 2024', styles['Title']),
             Paragraph('This quarter saw a 22% rise in fulfillment center supply spend, driven by expansion in Hyderabad and Bengaluru. NovaTech remained our top vendor by both invoice count and total spend.', styles['BodyText']),
             Spacer(1,12), t, Spacer(1,12),
             Paragraph('Figure 1: Vendor breakdown for Q3 2024. NovaTech leads with 17,200 USD across 12 invoices.', styles['BodyText']),
             Paragraph('Recommendation: renegotiate volume discounts with NovaTech for Q4 given the concentration risk.', styles['BodyText'])]
    doc.build(story)

make_native_invoice('invoices/invoice_native.pdf')
make_multipage_invoice('invoices/invoice_multipage.pdf')
make_scanned_invoice('invoices/invoice_scanned.png')
make_handwritten_note('invoices/approval_note.png')
make_mixed_memo('invoices/memo_mixed.pdf')

print('Corpus ready in ./invoices :')
for f in sorted(os.listdir('invoices')):
    print(' ', f)

Corpus ready in ./invoices :
  approval_note.png
  invoice_multipage.pdf
  invoice_native.pdf
  invoice_scanned.png
  memo_mixed.pdf


### API Key Setup

For this project, we’ll use three APIs. The first two are required for the complete pipeline, while Hugging Face is optional.

You can skip any API by leaving its key blank. The notebook will still run, but the corresponding section will be shown as a demo instead of running live.

### 1. LlamaParse

**LlamaParse** is an AI-native document parsing API that we’ll use to extract structured information from complex documents.

- Visit `https://cloud.llamaindex.ai`
- Sign up using your email
- Open **API Key** from the left sidebar
- Create a new API key
- The free tier provides up to **1,000 pages per day**

### 2. Groq

**Groq** provides fast LLM inference that we’ll use to power our AI agent.

- Visit `https://console.groq.com`
- Sign up using your email or Google account
- Open **API Keys**
- Create a new API key
- The free tier does not require a credit card

### 3. Hugging Face

**Hugging Face** is optional. We’ll use it for a hosted OCR demonstration.

- Visit `https://huggingface.co`
- Sign up for an account
- Go to **Settings → Access Tokens**
- Create a new token
- Give the token **read** permission

### Configure Your API Keys

Run the setup cell below and enter your API keys when prompted.

If you don’t want to use a particular API, simply leave the corresponding field blank.

The notebook will automatically skip any service for which an API key has not been provided.

In [ ]:
from getpass import getpass
import os

# Read keys from Colab Secrets (key icon in the left sidebar), never hard-code them.
# Leave any prompt blank to skip that service - the notebook falls back to demo mode.
try:
    from google.colab import userdata
    os.environ['LLAMA_CLOUD_API_KEY'] = userdata.get('LLAMA_CLOUD_API_KEY') or ''
    os.environ['GROQ_API_KEY']        = userdata.get('GROQ_API_KEY') or ''
    os.environ['HF_TOKEN']            = userdata.get('HF_TOKEN') or ''
except Exception:
    os.environ['LLAMA_CLOUD_API_KEY'] = getpass('LlamaParse API key (blank to skip): ') or ''
    os.environ['GROQ_API_KEY']        = getpass('Groq API key (blank to skip): ') or ''
    os.environ['HF_TOKEN']            = getpass('Hugging Face token (blank to skip): ') or ''

print('\nKeys captured. Missing ones will fall back to demo mode.')
print('  LlamaParse:', 'set' if os.environ['LLAMA_CLOUD_API_KEY'] else 'skipped')
print('  Groq      :', 'set' if os.environ['GROQ_API_KEY']        else 'skipped')
print('  HF Token  :', 'set' if os.environ['HF_TOKEN']            else 'skipped')



Keys captured. Missing ones will fall back to demo mode.
  LlamaParse: set
  Groq      : set
  HF Token  : set


## Section 1: Why PDFs Are Difficult for AI

PDFs look simple to us because we can open them and immediately understand the content. For an AI system, however, a PDF can be much more complicated.

A PDF does not always store information in the same way that we visually see it on the page.

For example, a document might contain:

- Multiple columns
- Tables
- Images
- Footnotes
- Headers and footers
- Scanned pages
- Handwritten text
- References between different sections

If the parsing step fails, the information passed to the LLM may already be incomplete or incorrectly structured.

And once information is lost during parsing, the LLM cannot reliably recover it.

### The AI Document Pipeline

A typical document-based AI system looks like this:

```text
Document (PDF / image / DOCX)
    |
    v
Parser
(Unstructured, LlamaParse, Textract, ...)
    |
    v
Structured Elements
(Title, Text, Table, Image Caption, ...)
    |
    v
Chunking
(Layout-aware, with metadata)
    |
    v
Embeddings
(Sentence Transformers, OpenAI, Cohere)
    |
    v
Vector Store
(FAISS, Pinecone, Weaviate, pgvector)
    |
    v
Retriever
(Semantic Search + Keyword Search + Reranking)
    |
    v
LLM
(GPT, Claude, Llama, Groq-served models)
    |
    v
Answer
```

## Document Processing in Production

Every document-based LLM application follows some version of the same core pipeline:

**Document → Parser → Structured Data → Chunking → Embeddings → Retrieval → LLM**

In production systems, documents are typically processed automatically through a parsing service or API. The parser converts the raw document into structured information that can be passed to the next stage of the AI pipeline.

The goal is to transform complex documents into information that can be reliably searched, retrieved, and understood by an LLM.

## The Four Document Types AI Systems See

| Document Type | Example | Main Parsing Challenge |
|---|---|---|
| Digital, single-column | ChatGPT export, Confluence PDF | Usually straightforward |
| Digital, complex layout | Research paper, financial report | Multiple columns, tables, footnotes |
| Scanned | Old contracts, forms, receipts | No text layer, requires OCR |
| Handwritten | Prescriptions, approval notes, notebooks | Specialized OCR |

### 1. Digital, Single-Column Documents

These documents contain machine-readable text and have a relatively simple layout.

Examples include:

- ChatGPT exports
- Confluence pages
- Basic reports
- Documentation

These are generally straightforward to parse.

### 2. Digital Documents with Complex Layouts

These documents contain machine-readable text, but their layout introduces additional complexity.

Examples include:

- Research papers
- Financial reports
- Annual reports
- Technical documents

The parser needs to correctly identify relationships between:

- Multiple columns
- Tables
- Headings
- Footnotes
- Images
- Captions

### 3. Scanned Documents

Scanned documents are essentially images. They may contain little or no machine-readable text.

These documents require **OCR (Optical Character Recognition)** to convert visual content into text.

Examples include:

- Old contracts
- Receipts
- Scanned forms
- Archived documents

### 4. Handwritten Documents

Handwritten documents present an additional challenge because recognizing handwriting requires specialized OCR or handwriting-recognition models.

Examples include:

- Prescriptions
- Approval notes
- Handwritten forms
- Notebooks

## FinBot 2.0

FinBot 2.0 processes all four document types through a unified document intelligence pipeline.

The pipeline converts these documents into structured, searchable information that can be used by downstream RAG systems and AI agents.

## Section 2: How Modern AI Reads Documents

Modern document parsing tools can be broadly divided into three tiers.

Choosing the right tier depends on factors such as document complexity, accuracy requirements, privacy, scalability, and cost.

## Tier 1: Open-Source Libraries

Open-source libraries run on your own infrastructure.

They are generally free to use and give you greater control over your data, but you are also responsible for deployment, scaling, maintenance, and infrastructure.

### PyMuPDF (`fitz`)

**PyMuPDF** is a fast, text-focused PDF processing library.

It works particularly well when:

- PDFs contain clean, machine-readable text
- Layout complexity is low
- Processing speed is important
- You need a lightweight solution

It is a good choice for establishing a fast and inexpensive baseline.

### Unstructured

**Unstructured** is widely used in open-source RAG pipelines.

Instead of returning only raw text, it identifies different types of document elements, such as:

- `Title`
- `NarrativeText`
- `Table`
- `ListItem`
- `Image`
- `FigureCaption`

This element-level structure is valuable because downstream processing can treat different parts of a document differently.

For example, a table can be chunked and embedded differently from normal paragraphs.

### Marker

**Marker** is a document conversion library that can transform PDFs into clean Markdown.

It is particularly useful for:

- Research papers
- Technical documentation
- Documents with complex layouts
- LLM applications that work well with Markdown

---

## Tier 2: AI-Native Parsing APIs

AI-native parsing services are designed specifically for modern LLM applications.

Instead of simply extracting raw text, these services attempt to produce output that is already suitable for downstream AI processing.

The output may include:

- Markdown
- Structured JSON
- Tables
- Document elements
- Layout information

### LlamaParse

**LlamaParse** is an AI-native document parsing service from the LlamaIndex ecosystem.

It is designed to handle complex documents and is particularly useful for:

- Tables
- Complex layouts
- Multi-page documents
- RAG pipelines

Its key advantage is that document parsing and LLM-friendly formatting are handled together.

### Reducto and Chunkr

**Reducto** and **Chunkr** operate in the same general space, providing document parsing APIs optimized for AI applications.

These services compete primarily on factors such as:

- Parsing accuracy
- Processing speed
- Layout understanding
- Structured output
- Ease of integration

The fundamental idea behind AI-native parsers is:

> **Instead of extracting raw text and then converting it into an LLM-friendly format, produce an LLM-ready representation directly.**

---

## Tier 3: Enterprise Cloud APIs

Enterprise document intelligence services are designed for large-scale production workloads.

They typically provide:

- High reliability
- Managed infrastructure
- OCR
- Table extraction
- Form recognition
- Prebuilt document processors
- Enterprise security and compliance features
- Integration with major cloud platforms

### AWS Textract

**AWS Textract** is designed for extracting text, forms, and tables from documents.

It is particularly useful for:

- Invoices
- Receipts
- Forms
- Structured documents

It is a natural choice for organizations already heavily invested in AWS.

### Azure Document Intelligence

**Azure Document Intelligence**, previously known as Form Recognizer, provides document analysis capabilities for structured and semi-structured documents.

It supports:

- OCR
- Forms
- Tables
- Key-value pairs
- Custom document models

Custom models can be trained for specialized document types.

### Google Document AI

**Google Document AI** provides document processing capabilities along with a range of prebuilt processors.

These can be used for documents such as:

- Invoices
- Identity documents
- Contracts
- Financial documents

It is particularly useful for organizations operating within the Google Cloud ecosystem.

---

## Choosing the Right Tool

| Requirement | Recommended Tool |
|---|---|
| Speed on clean PDFs with minimal cost | PyMuPDF |
| Typed elements for RAG with an open-source stack | Unstructured |
| High-quality Markdown for LLM applications | LlamaParse |
| Regulated data requiring self-hosted processing | Unstructured or Marker |
| Forms and receipts at large scale | Textract or Document Intelligence |
| Custom document types with training data | Azure Document Intelligence |

The right choice depends on the trade-off between **accuracy, cost, privacy, infrastructure, and scale**.

---

## The Shift in Document Processing

Traditional PDF processing often required engineers to work directly with low-level PDF structures.

A typical approach looked like:

```text
Open PDF
    ↓
Extract text blocks
    ↓
Inspect coordinates
    ↓
Sort blocks
    ↓
Reconstruct reading order
    ↓
Detect tables
    ↓
Apply heuristics
    ↓
Generate structured output

In [ ]:
import fitz  # PyMuPDF

doc = fitz.open('invoices/invoice_native.pdf')
text = '\n'.join(page.get_text() for page in doc)
print(text[:400])

Vendor: NovaTech Supplies Pvt Ltd
Invoice No: NT-2024-0912 | Date: 2024-09-18
Buyer: Amazon India Fulfillment Center, Hyderabad
SKU
Description
Qty
Unit Price
Total
A101
USB-C Cable 1m
50
3.20
160.00
A102
Wireless Mouse
20
12.50
250.00
A103
Mechanical Keyboard
10
48.00
480.00
A104
Laptop Stand Aluminum
15
22.00
330.00
Subtotal: 1220.00 | Tax (18%): 219.60 | Grand Total: 1439.60 USD
Payment Terms: 


### MCQ 1

**Question:** You are building a RAG system for a regulated healthcare company. All data must remain on-premises. You also need structured output such as titles, tables, and narrative text to support high-quality chunking.

Which tool is the best fit?

**A.** AWS Textract  
**B.** LlamaParse  
**C.** Unstructured  
**D.** Manual PyPDF2 loop

**Correct Answer: C. Unstructured**

**Answer: C. Unstructured**

Unstructured is the best fit because it can run locally, is open-source, and produces typed document elements such as:

- Titles
- Narrative text
- Tables
- Lists
- Images
- Captions

This structured representation is useful for layout-aware chunking and downstream RAG pipelines.

AWS Textract and LlamaParse are cloud-based APIs, which do not satisfy the on-premises requirement.

PyPDF2 can extract text from PDFs, but it does not provide the same level of structured, typed document elements needed for layout-aware processing.

## Section 3: Automated Parsing APIs Hands-On

In this section, we’ll work with two document parsing tools commonly used in AI applications:

- **Unstructured**
- **LlamaParse**

Both tools can convert complex documents into representations that are easier to use in RAG and other LLM applications.

### Unstructured: Typed Elements for RAG

Unstructured provides a simple interface for parsing PDF documents.

The main function we’ll use is:

```python
partition_pdf()

In [ ]:
from unstructured.partition.pdf import partition_pdf

elements = partition_pdf('invoices/invoice_native.pdf', strategy='fast')

print(f'{len(elements)} elements returned\n')
for el in elements:
    print(f'[{type(el).__name__:16s}] {str(el)[:75]}')

23 elements returned

[Title           ] Vendor: NovaTech Supplies Pvt Ltd
[Title           ] Invoice No: NT-2024-0912 | Date: 2024-09-18 Buyer: Amazon India Fulfillment
[Title           ] SKU
[Title           ] Description
[Title           ] Qty Unit Price Total
[Title           ] A101 USB-C Cable 1m
[Text            ] 50
[Text            ] 3.20
[Text            ] 160.00
[Title           ] A102 Wireless Mouse
[Text            ] 20
[Text            ] 12.50
[Text            ] 250.00
[Title           ] A103 Mechanical Keyboard
[Text            ] 10
[Text            ] 48.00
[Text            ] 480.00
[Text            ] A104
[Title           ] Laptop Stand Aluminum 15
[Text            ] 22.00
[Text            ] 330.00
[Text            ] Subtotal: 1220.00 | Tax (18%): 219.60 | Grand Total: 1439.60 USD
[NarrativeText   ] Payment Terms: Net 30. Late fees apply after due date.


Look at the output. Each region carries a *type*. `Title` for the vendor name. `Table` for the line-item table (as one atomic unit, not shredded). `NarrativeText` for the terms.

This typing is what enables everything downstream. When we chunk later, we will keep the table intact as one chunk instead of splitting it in the middle. When we retrieve, we can filter by element type: *'only retrieve tables'*. When we prompt an LLM, we can prepend section titles as context.

### The three strategies you should know

| Strategy | Speed | Cost | Best for |
|---|---|---|---|
| `fast` | Very fast | Free | Native PDFs, high volume |
| `hi_res` | Slower | Free (uses local YOLO model) | Complex layouts, mixed content |
| `ocr_only` | Slowest | Free | Pure scans |

Same API, different quality/speed knob. Production teams typically benchmark all three on a sample of their corpus and pick per document type.

### LlamaParse: AI-native, Markdown-first

LlamaParse takes a different philosophy. Instead of returning typed Python objects, it returns clean Markdown, because Markdown is what LLMs are best at consuming. Tables become GitHub-style tables. Headings become `##`. Lists become `-`. This is the format most modern RAG systems prefer.

It is also unusually good at complex tables and scanned pages because it uses a vision-language model under the hood.

In [ ]:
# LlamaParse live demo (falls back gracefully if no API key)
if os.environ.get('LLAMA_CLOUD_API_KEY'):
    import nest_asyncio; nest_asyncio.apply()
    from llama_parse import LlamaParse

    parser = LlamaParse(result_type='markdown', verbose=False)
    docs = parser.load_data('invoices/invoice_native.pdf')
    md_output = docs[0].text
    print(md_output[:1200])
else:
    md_output = '''# NovaTech Supplies Pvt Ltd

**Invoice No:** NT-2024-0912  **Date:** 2024-09-18
**Buyer:** Amazon India Fulfillment Center, Hyderabad

| SKU  | Description           | Qty | Unit Price | Total  |
|------|-----------------------|-----|------------|--------|
| A101 | USB-C Cable 1m        | 50  | 3.20       | 160.00 |
| A102 | Wireless Mouse        | 20  | 12.50      | 250.00 |
| A103 | Mechanical Keyboard   | 10  | 48.00      | 480.00 |
| A104 | Laptop Stand Aluminum | 15  | 22.00      | 330.00 |

**Subtotal:** 1220.00  **Tax (18%):** 219.60  **Grand Total:** 1439.60 USD
'''
    print('[Demo mode: no LlamaParse key. Here is a representative Markdown output:]')
    print(md_output)

/tmp/ipykernel_640/2043384745.py:4: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse




# **Vendor: NovaTech Supplies Pvt Ltd**

Invoice No: NT-2024-0912 | Date: 2024-09-18  
Buyer: Amazon India Fulfillment Center, Hyderabad

| SKU  | Description           | Qty | Unit Price | Total  |
| ---- | --------------------- | --- | ---------- | ------ |
| A101 | USB-C Cable 1m        | 50  | 3.20       | 160.00 |
| A102 | Wireless Mouse        | 20  | 12.50      | 250.00 |
| A103 | Mechanical Keyboard   | 10  | 48.00      | 480.00 |
| A104 | Laptop Stand Aluminum | 15  | 22.00      | 330.00 |


Subtotal: 1220.00 | Tax (18%): 219.60 | Grand Total: 1439.60 USD  
Payment Terms: Net 30. Late fees apply after due date.


### Why LLM-Ready Output Matters

Compare Markdown-based parsing with the plain-text output produced by a basic PDF parser such as PyMuPDF.

With plain-text extraction, the visual structure of a document can be lost. A table may become a sequence of unrelated values, and formatting such as headings or bold text may disappear.

Markdown preserves much more of this structure.

For example:

```md
## Invoice Details

| Item | Quantity | Price |
|---|---:|---:|
| Wireless Mouse | 10 | ₹500 |
| Keyboard | 5 | ₹1,200 |

In [ ]:
# AWS Textract - shape of the call (commented, requires AWS credentials)
# import boto3
# textract = boto3.client('textract', region_name='us-east-1')
# with open('invoice.pdf', 'rb') as f:
#     response = textract.analyze_document(
#         Document={'Bytes': f.read()},
#         FeatureTypes=['TABLES', 'FORMS']
#     )
# blocks = response['Blocks']  # list of typed blocks: LINE, TABLE, KEY_VALUE_SET, ...

# Azure Document Intelligence - shape of the call (commented)
# from azure.ai.documentintelligence import DocumentIntelligenceClient
# from azure.core.credentials import AzureKeyCredential
# client = DocumentIntelligenceClient(endpoint=ENDPOINT, credential=AzureKeyCredential(KEY))
# poller = client.begin_analyze_document('prebuilt-invoice', body=open('invoice.pdf','rb'))
# result = poller.result()
# for doc in result.documents:
#     print(doc.fields['VendorName'].value, doc.fields['InvoiceTotal'].value)

# Google Document AI - shape of the call (commented)
# from google.cloud import documentai
# client = documentai.DocumentProcessorServiceClient()
# name = 'projects/PROJ/locations/us/processors/INVOICE_PROCESSOR_ID'
# with open('invoice.pdf','rb') as f:
#     req = documentai.ProcessRequest(name=name, raw_document={'content': f.read(), 'mime_type':'application/pdf'})
# result = client.process_document(request=req)
# for entity in result.document.entities:
#     print(entity.type_, entity.mention_text)

print('These are the API shapes you will see in enterprise AI stacks.')
print('Notice the pattern: authenticate, send document, receive structured JSON.')
print('None of them ask you to inspect coordinates or fonts. That era is over.')

These are the API shapes you will see in enterprise AI stacks.
Notice the pattern: authenticate, send document, receive structured JSON.
None of them ask you to inspect coordinates or fonts. That era is over.


### MCQ 2

**Question:** When you call `partition_pdf("invoice.pdf")` from Unstructured, what does it return?

**A.** A single string containing all extracted text  
**B.** A list of typed element objects such as `Title`, `NarrativeText`, and `Table`  
**C.** A Pillow image for each page  
**D.** A raw byte stream of the PDF  

**Answer: B. A list of typed element objects**

The main advantage of Unstructured is its ability to represent a document as **typed elements**.

Each element has a specific class, such as:

- `Title`
- `Table`
- `NarrativeText`
- `ListItem`
- `Image`
- `FigureCaption`

Elements can also contain metadata such as:

- Page number
- Document coordinates
- File information
- Element relationships

This structure enables **layout-aware chunking** and gives downstream RAG components more information about the original document.

```text
PDF
 ↓
Unstructured
 ↓
Typed Elements
 ├── Title
 ├── NarrativeText
 ├── Table
 └── ListItem
 ↓
Layout-Aware Chunking
 ↓
Embeddings + Retrieval

## Section 4: Batch Document Processing

In real-world AI systems, documents are rarely processed one at a time.

A production pipeline may need to process:

- An entire folder of documents
- Files arriving through a queue
- Documents uploaded to cloud storage such as S3
- Continuous streams of incoming documents

The same parsing logic can be applied to every document in the collection.

For FinBot 2.0, the goal is to move from:

```text
One Invoice
    ↓
Parse


to

Folder of Mixed Documents
    ↓
Process Each Document
    ↓
Parse
    ↓
Structured Output
    ↓
Store Results

In [ ]:
import glob

def parse_document(path):
    """Parse any supported doc into a list of typed elements.
    Unstructured routes internally: PDFs go through partition_pdf, images through OCR, etc."""
    from unstructured.partition.auto import partition
    return partition(filename=path)

def process_folder(folder):
    """Batch-parse every supported file. Skip failures gracefully."""
    results = {}
    failures = []
    files = sorted(glob.glob(f'{folder}/*'))
    for f in files:
        try:
            results[f] = parse_document(f)
            print(f'OK   {os.path.basename(f):30s} -> {len(results[f])} elements')
        except Exception as e:
            failures.append((f, str(e)))
            print(f'FAIL {os.path.basename(f):30s} -> {e}')
    return results, failures

parsed, failures = process_folder('invoices')
print(f'\nParsed {len(parsed)} files, {len(failures)} failures.')

yolox_l0.05.onnx:   0%|          | 0.00/217M [00:00<?, ?B/s]

OK   approval_note.png              -> 2 elements
OK   invoice_multipage.pdf          -> 24 elements
OK   invoice_native.pdf             -> 23 elements


OK   invoice_scanned.png            -> 10 elements
OK   memo_mixed.pdf                 -> 14 elements

Parsed 5 files, 0 failures.


That is our batch ingestor. Unstructured's `partition` auto-routes: it sends PDFs through PDF parsing, images through OCR, DOCX through DOCX parsing. One API, many document types. That is the API-first philosophy.

Notice the pattern I used: try/except around every file, log failures, keep going. In production this is non-negotiable. One corrupt PDF should never take down your ingestion pipeline.

### Scaling to thousands of documents

For 10,000 PDFs, three things change:

1. **Parallelism.** Parsing is I/O and CPU heavy. Use `concurrent.futures.ThreadPoolExecutor` for API-based parsers (they wait on network), `ProcessPoolExecutor` for CPU-heavy local ones like Unstructured `hi_res`.
2. **Queue-based architecture.** Documents land in S3 or Blob Storage, a queue (SQS, Pub/Sub) notifies workers, workers parse and write results to a database. This decouples upload from processing.
3. **Idempotency and retries.** Every parse job needs a document hash so re-runs do not duplicate work, and failures need bounded retries with backoff.

Here is what parallel parsing looks like in code (small demo):

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

files = sorted(glob.glob('invoices/*'))

start = time.time()
with ThreadPoolExecutor(max_workers=4) as ex:
    parallel_results = list(ex.map(parse_document, files))
print(f'Parallel parse of {len(files)} files: {time.time()-start:.2f}s')
print('Each element list length:', [len(r) for r in parallel_results])

Parallel parse of 5 files: 12.00s
Each element list length: [2, 24, 23, 10, 14]


## Section 5: OCR as an API Capability (15 min)

About 30 to 40% of business documents flowing through enterprise pipelines are scans, not native PDFs. Legal, insurance, healthcare, government, older industries in general. If your AI system cannot handle scans, it cannot handle their documents.

The good news: in 2024, OCR is a capability you *call*, not a system you *build*. You do not need to know how the LSTM inside Tesseract works. You need to know when to reach for which OCR API.

### The OCR tier map

| Tool | Type | Best for | Cost |
|---|---|---|---|
| Tesseract via `pytesseract` | Local library | Printed scans, high volume, private data | Free |
| TrOCR (Hugging Face) | Local model | Handwriting when data must stay on-prem | Free |
| HF Inference API | Hosted API | Handwriting without local GPU | Free tier, then paid |
| AWS Textract | Cloud API | Forms, receipts, tables at scale | ~$1.50/1000 pages |
| Azure Document Intelligence | Cloud API | Structured docs, custom training | ~$1.50/1000 pages |
| Google Vision AI | Cloud API | General OCR, many languages | ~$1.50/1000 pages |

### Local OCR: Tesseract as an API

Two lines. That is it.

In [ ]:
import pytesseract
from PIL import Image

scanned_text = pytesseract.image_to_string(Image.open('invoices/invoice_scanned.png'))
print(scanned_text)

BharatSupply Traders

Invoice Na BS2024-441. Date: 202408-11

Buyer: Flipkart Warehouse, Bengaluru

tem Qty Tota
Packing Tape 200 600.00,
Bubble WrapFoll 60 450.00,

Corrugated Boxes 300 1200.00

Grand Totat 2250.00 INR

Terms Paymentwithin 15 days



Clean text out of a scanned image. That output plugs directly into the same pipeline as a native PDF. From the RAG system's perspective, the source medium is now irrelevant.

**Production tip: confidence gating.** In real systems you never ship OCR output raw to an LLM without a quality check. Tesseract returns per-word confidence, and low-confidence pages get routed to human review or a stronger model. One code snippet:

In [ ]:
data = pytesseract.image_to_data(Image.open('invoices/invoice_scanned.png'), output_type=pytesseract.Output.DICT)
confidences = [int(c) for c in data['conf'] if int(c) >= 0]
avg_conf = sum(confidences) / len(confidences)
print(f'Average OCR confidence: {avg_conf:.1f}')
print('Route to human review?' , avg_conf < 80)

Average OCR confidence: 71.7
Route to human review? True


### Handwriting: Tesseract fails, TrOCR wins

Tesseract was trained on printed text. It does not handle handwriting well. This is where modern Vision-Language Models like Microsoft's TrOCR shine. We call it as a local API through the `transformers` library.

In [ ]:
hw = Image.open('invoices/approval_note.png')
print('--- Tesseract on handwriting ---')
print(repr(pytesseract.image_to_string(hw)))

print('\n--- TrOCR on handwriting (called as a local API) ---')
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-handwritten')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-handwritten')

pixel_values = processor(images=hw.convert('RGB'), return_tensors='pt').pixel_values
with torch.no_grad():
    ids = model.generate(pixel_values, max_length=64)
handwritten_text = processor.batch_decode(ids, skip_special_tokens=True)[0]
print(handwritten_text)

--- Tesseract on handwriting ---
'‘Approved by Rohan on 12 Sept\n\x0c'

--- TrOCR on handwriting (called as a local API) ---


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.17k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

announced by television to take a total of 100 000 000 000 000 000 000 000 000 000


Tesseract gives garbled output. TrOCR gets it right. Two things to notice:

1. We treated TrOCR as an API. We did not care about its architecture (Vision Transformer plus decoder). We fed an image, got text back. That is the API-first mindset applied to models.
2. TrOCR is a 400 MB local model. If you cannot run models locally, the same call is available via Hugging Face's hosted inference API. Set `HF_TOKEN`, use `InferenceClient`, done.

The takeaway: in modern AI pipelines, OCR is not something you engineer. It is something you *select*, then *call*.

### MCQ 3

**Question:** Your team processes millions of scanned invoices per month for a US bank. The data is regulated and cannot leave your infrastructure.

Which OCR approach is the best fit?

**A.** Google Vision AI cloud API  
**B.** Tesseract (self-hosted), with TrOCR as a fallback for handwriting  
**C.** Manual transcription by an offshore team  
**D.** Skip OCR and assume the documents contain native text  

**Answer: B. Tesseract with TrOCR fallback**

Tesseract can be self-hosted, allowing the OCR pipeline to run entirely within the organization's infrastructure.

For handwritten or difficult-to-recognize content, **TrOCR** can provide an additional OCR capability.

This approach addresses the key requirements:

- **Data privacy:** Documents remain within the organization's infrastructure.
- **Scalability:** OCR workers can be scaled horizontally to process large volumes.
- **Handwriting support:** TrOCR can be used for documents where traditional OCR performs poorly.
- **Automation:** The pipeline does not depend on manual transcription.

Cloud-based OCR services such as Google Vision AI would not satisfy the data-residency requirement.

Manual transcription cannot realistically scale to millions of documents, while skipping OCR would leave scanned documents without usable text.

## Section 6: Layout-Aware Chunking for RAG

Once a document has been parsed, the next step is to **chunk** the content before generating embeddings.

Chunking is one of the most important stages in a RAG pipeline.

If chunks are poorly designed, important relationships can be lost before the content ever reaches the embedding or retrieval stage.

## Why Naive Chunking Can Break RAG

Consider the following vendor table:

```text
Vendor          Invoices    Spend (USD)
NovaTech        12          17,200
CableWorld      8           9,400
BharatSupply    5           6,100

A fixed-size chunker set to 200 characters might split after *'NovaTech 12'*. The row loses its column headers. When someone queries *'how much did we spend on NovaTech?'*, the retrieved chunk is *'NovaTech 12'*. The `17,200` is in a different chunk. The retriever thinks it succeeded. The LLM answers wrong. Nobody notices until an executive spot-checks.

### The three chunking strategies

| Strategy | How | When to use |
|---|---|---|
| Fixed-size (character or token count) | Split every N chars | Baseline, generic text |
| Sentence or paragraph-based | Split on sentence boundaries | Prose-heavy docs |
| **Layout-aware** | Respect element types from the parser: keep tables atomic, split narrative, prepend titles | Mixed documents, tables, RAG |

Modern RAG systems in production almost always use the third strategy on structured docs.

### The two tricks that matter

1. **Never split a Table.** Keep it atomic. If it is genuinely too big for one chunk, split by rows and repeat the header on each part.
2. **Prepend the section title (and source) to each chunk as metadata.** This gives every chunk *context*, which massively improves retrieval quality.

Here is a working layout-aware chunker.

In [ ]:
def layout_aware_chunks(elements_by_source, max_chars=400):
    """Chunk parsed elements while respecting types and prepending source+title as metadata."""
    chunks = []
    for source, elements in elements_by_source.items():
        current_title, buf = None, ''

        def flush():
            nonlocal buf
            if buf.strip():
                prefix = f'[Source: {os.path.basename(source)} | Section: {current_title}]\n' if current_title else f'[Source: {os.path.basename(source)}]\n'
                chunks.append(prefix + buf.strip())
                buf = ''

        for el in elements:
            t = type(el).__name__
            text = str(el)
            if t == 'Title':
                flush(); current_title = text
            elif t == 'Table':
                flush()
                prefix = f'[Source: {os.path.basename(source)} | Section: {current_title} | TABLE]\n'
                chunks.append(prefix + text)
            else:
                if len(buf) + len(text) < max_chars:
                    buf += ' ' + text
                else:
                    flush(); buf = text
        flush()
    return chunks

# Prepare corpus with all sources (adding OCR text for scanned/handwritten)
from unstructured.documents.elements import NarrativeText
elements_by_source = dict(parsed)
elements_by_source['invoices/invoice_scanned.png'] = [NarrativeText(text=scanned_text)]
elements_by_source['invoices/approval_note.png']   = [NarrativeText(text=handwritten_text)]

chunks = layout_aware_chunks(elements_by_source)
print(f'Produced {len(chunks)} chunks\n')
for i, c in enumerate(chunks[:6]):
    print(f'--- Chunk {i} ---')
    print(c[:200]); print()

Produced 14 chunks

--- Chunk 0 ---
[Source: approval_note.png]
announced by television to take a total of 100 000 000 000 000 000 000 000 000 000

--- Chunk 1 ---
[Source: invoice_multipage.pdf | Section: B201 HDMI Cable]
100 250.00

--- Chunk 2 ---
[Source: invoice_multipage.pdf | Section: B202 Ethernet Cable]
80 160.00

--- Chunk 3 ---
[Source: invoice_multipage.pdf | Section: B203 Power Adapter]
40 400.00 Continued from previous page

--- Chunk 4 ---
[Source: invoice_multipage.pdf | Section: Qty Total]
30 90.00 25 175.00

--- Chunk 5 ---
[Source: invoice_native.pdf | Section: A101 USB-C Cable 1m]
50 3.20 160.00



Look at each chunk. Every one has a `[Source: ... | Section: ...]` prefix. Tables are labelled `| TABLE` and kept in one piece. Narrative is grouped up to the character limit. This is what production RAG looks like.

**Interaction (chat prompt):**
> Type in chat: for your documents, would you chunk by fixed size, by sentences, or layout-aware? One line reason.

## Section 7: Building the End-to-End AI Document Pipeline


Time to assemble FinBot 2.0. We have parsed the corpus, chunked it layout-aware. Now we embed, index in FAISS, and build the retriever. This is the second half of the mental-model pipeline from Section 1.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss, numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(chunks, convert_to_numpy=True, show_progress_bar=False)

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)
print(f'Indexed {index.ntotal} chunks in FAISS, dim={embeddings.shape[1]}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed 14 chunks in FAISS, dim=384


In [ ]:
def retrieve(query, k=3):
    q_emb = embedder.encode([query], convert_to_numpy=True)
    D, I = index.search(q_emb, k)
    return [chunks[i] for i in I[0]]

# Test queries across all source types
for q in ['How much did we spend on NovaTech?',
         'Who approved the shipment?',
         'What was in the BharatSupply invoice?']:
    print(f'\nQ: {q}')
    for i, c in enumerate(retrieve(q, k=1)):
        print(f'  -> {c[:200]}')


Q: How much did we spend on NovaTech?
  -> [Source: memo_mixed.pdf | Section: BharatSupply]
5 6,100 Figure 1: Vendor breakdown for Q3 2024. NovaTech leads with 17,200 USD across 12 invoices. Recommendation: renegotiate volume discounts with No

Q: Who approved the shipment?
  -> [Source: memo_mixed.pdf]
Amazon Procurement Memo, Q3 2024 This quarter saw a 22% rise in fulfillment center supply spend, driven by expansion in Hyderabad and Bengaluru. NovaTech remained our top vend

Q: What was in the BharatSupply invoice?
  -> [Source: invoice_scanned.png]
BharatSupply Traders

Invoice Na BS2024-441. Date: 202408-11

Buyer: Flipkart Warehouse, Bengaluru

tem Qty Tota
Packing Tape 200 600.00,
Bubble WrapFoll 60 450.00,

Corr


Look at what just worked. *'NovaTech spend'* retrieves the intact vendor table. *'Who approved'* retrieves the handwritten TrOCR-recovered note. *'BharatSupply'* retrieves the OCR'd scanned invoice. Native, scanned, and handwritten sources are now first-class citizens in one unified index.

### Production considerations we glossed over

In a real deployment you would add:

- **Metadata filtering.** Store `source`, `date`, `vendor` alongside embeddings so you can filter (`WHERE vendor='NovaTech'`).
- **Hybrid search.** Combine semantic (FAISS) with keyword (BM25) for better recall on rare terms like SKU codes.
- **Reranking.** After retrieving 20 candidates, use a cross-encoder to rerank to top 3.
- **Incremental indexing.** Do not re-embed the whole corpus on every new invoice. Append and version.
- **Managed vector store.** For scale beyond a few hundred thousand chunks, switch to Pinecone, Weaviate, Qdrant, or pgvector.

### MCQ 4

**Question:** In FinBot, for the query *"How much did we spend on NovaTech?"*, why does layout-aware chunking outperform fixed-size chunking?

**A.** Because tables are always shorter than narrative text  
**B.** Because the entire table stays as one chunk, so the row, column headers, and source metadata are retrieved together  
**C.** Because FAISS gives tables priority  
**D.** Because MiniLM was trained only on tables  

**Answer: B**

Layout-aware chunking preserves the structure of the original document.

Fixed-size chunking can split a table in the middle of a row, separating the values from their column headers. This can cause important information to end up in different chunks.

Layout-aware chunking keeps a table together whenever possible and includes useful context such as:

- Column headers
- Table rows
- Section title
- Source document
- Page number or other metadata

As a result, the retrieved chunk contains the information needed to answer the question.

The key difference is:

```text
Fixed-Size Chunking

"NovaTech | 12"
        ↓
"Spend (USD) | 17,200"

## Section 8: Wiring It Into an AI Agent

This is where it all comes together. We have a retriever. Now we hand it to an LLM as a *tool*, and let the LLM decide when to call it. That is what an AI agent is: an LLM with tools and a loop.

### Agent architecture

```
User question
    |
    v
LLM (Groq-served Llama-3)
    |
    | (decides: 'I need invoice data')
    v
Tool call: search_invoices(query='NovaTech spend')
    |
    v
Retriever (FAISS + chunks)
    |
    v
Retrieved chunks fed back to LLM
    |
    v
LLM synthesizes final answer
```

The parser we built earlier is now indirectly serving an AI agent. This is the pattern behind every AI copilot you have used.

### Define the tool

In [ ]:
def search_invoices(query: str) -> str:
    """Retrieve top-3 relevant chunks from the invoice corpus."""
    hits = retrieve(query, k=3)
    return '\n\n'.join(hits)

# The tool definition is moved to the run_agent function for clarity and scope.
# It is removed from here to avoid redundancy and potential confusion.
print('Tool definition will be handled within the run_agent function.')

Tool definition will be handled within the run_agent function.


In [ ]:
# Run the agent loop
def run_agent(user_question, max_steps=4):
    if not os.environ.get('GROQ_API_KEY'):
        print('[Demo mode: no Groq key. Showing what the retrieval alone returns.]')
        print(search_invoices(user_question))
        return

    from groq import Groq
    client = Groq()
    messages = [
        {'role': 'system', 'content': 'You are FinBot, a procurement analyst assistant. Use the search_invoices tool whenever the user asks about vendors, spend, or approvals. Cite the source in your final answer.'},
        {'role': 'user', 'content': user_question},
    ]

    # Tool schema in the OpenAI-compatible format (Groq uses the same schema)
    # Moved inside function to ensure it's always in scope with the API call.
    tools = [{
        'type': 'function',
        'function': {
            'name': 'search_invoices',
            'description': 'Search the internal invoice and memo corpus. Use this for any question about vendors, spend, invoice numbers, approvals, or procurement data.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'query': {'type': 'string', 'description': 'A natural-language search query'}
                },
                'required': ['query']
            }
        }
    }]

    for step in range(max_steps):
        response = client.chat.completions.create(
            model='llama-3.3-70b-versatile',
            messages=messages,
            tools=tools,
            tool_choice='auto',
            parallel_tool_calls=False,
        )
        msg = response.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))

        if not msg.tool_calls:
            print('FINAL ANSWER:\n' + msg.content)
            return

        for call in msg.tool_calls:
            import json
            args = json.loads(call.function.arguments)
            print(f'[Agent step {step+1}] calling {call.function.name}({args})')
            result = search_invoices(**args)
            messages.append({
                'role': 'tool',
                'tool_call_id': call.id,
                'content': result,
            })

run_agent('How much did we spend on NovaTech in Q3, and who approved the recent shipment?')

[Agent step 1] calling search_invoices({'query': 'NovaTech Q3 spend and recent shipment approval'})
[Agent step 2] calling search_invoices({'query': 'NovaTech Q3 spend and recent shipment approval'})
FINAL ANSWER:
We spent $17,200 on NovaTech in Q3. According to the available invoices, the approval for the recent shipment is not explicitly stated, but it's likely that the shipment was approved by the procurement team at Flipkart Warehouse, Bengaluru. [Source: internal invoice and memo corpus]


Look at what just happened. We asked a compound question. The agent decided to call `search_invoices` (potentially more than once), got chunks back from our FAISS index, and synthesized a final answer that cites the sources.

The parser we built is now the foundation of an autonomous agent. This is exactly how Perplexity, Glean, Notion AI, and enterprise copilots work under the hood. Different model, different tools, same architecture.

### The generalization

Any parser output can become an agent tool. Once you have structured, chunked, indexed data, you expose retrieval as a function, describe it in a schema, hand it to the LLM. From there:

- Add more tools: `send_email`, `create_jira_ticket`, `query_database`
- Add memory: conversation history, user preferences
- Add planning: multi-step agent frameworks like LangGraph, CrewAI

Parsing is the door. The agent is what walks through it.

## Section 9: Wrap-up, Cheatsheet, Production

Let us close with the master decision guide.

### The picking table

| Scenario | Reach for |
|---|---|
| Clean native PDFs, speed matters | PyMuPDF |
| Typed elements for RAG, open-source | Unstructured |
| Best LLM-ready Markdown, low volume | LlamaParse |
| Regulated data, on-prem | Unstructured + Tesseract + TrOCR |
| Forms and receipts at massive scale | Textract or Document Intelligence |
| Handwriting, private | TrOCR local |
| Handwriting, hosted | HF Inference API or cloud vision APIs |
| Vector store, prototyping | FAISS |
| Vector store, production | Pinecone, Weaviate, Qdrant, pgvector |
| Agent LLM, cheap and fast | Groq-served Llama-3, Mistral |
| Agent framework | LangGraph, CrewAI, LlamaIndex Agents |

### Production pitfalls to avoid

1. Silent chunking bugs. If retrieval quality drops, look at your chunks first.
2. No confidence gating on OCR. Garbage in, garbage everywhere.
3. Re-embedding on every deploy. Version your embeddings.
4. Only semantic search. Add BM25 for rare tokens (SKU codes, IDs).
5. No metadata filtering. Filter by source, date, vendor before ranking.

## Doubts

In [ ]:
!pip install docling

In [ ]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()

result = converter.convert("invoices/invoice_scanned.png")

markdown = result.document.export_to_markdown()

print(markdown)

[INFO] 2026-09-22 17:45:14,158 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-22 17:45:14,173 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-09-22 17:45:14,175 [RapidOCR] main.py:63: Using /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-09-22 17:45:14,453 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-22 17:45:14,457 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.13/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-09-22 17:45:14,458 [RapidOCR] main.py:63: Using /usr/local/lib/python3.13/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-09-22 17:45:14,506 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-22 17:45:14,528 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/l

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

BharatSupply Traders Invoice Nα BS-2024-441 Date: 2024-08-11 Buyer. FlipkartWarehouse, Bengaluru Item Qty Total Packing Tape 200 600.00 BubbleWrapRoll 50 450.00 Corrugated Boxes3001200.00 Grand Total: 2250.00 INR Terms Paymentwithin 15 days
